In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
!git clone https://github.com/darklord-09/my_cuda_project.git /content/drive/MyDrive/cuda-project

fatal: destination path '/content/drive/MyDrive/cuda-project' already exists and is not an empty directory.


In [3]:
%cd /content/drive/MyDrive/cuda-project

/content/drive/MyDrive/cuda-project


Kernel building starts

In [ ]:
!nvidia-smi

Thu Aug 27 18:11:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%writefile vector_add.cu
#include <cstdio>
#include <cstdlib>

__global__ void vecAdd(const float *a,const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n) {
    c[i] = a[i] + b[i];
  }
}

int main(){
     int n = 1<<20;
     size_t bytes = n * sizeof(float);
     float *h_a, *h_b, *h_c;
     float *d_a, *d_b, *d_c;

     h_a = (float *)malloc(bytes);
     h_b = (float *)malloc(bytes);
     h_c = (float *)malloc(bytes);


     for(int i=0;i<n;i++){
       h_a[i]=1;
       h_b[i]=2;
     }

     cudaMalloc(&d_a, bytes);
     cudaMalloc(&d_b, bytes);
     cudaMalloc(&d_c, bytes);


     cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
     cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);


     int number_of_threads=1024; //per block threads
     int number_of_blocks=(n+number_of_threads-1)/number_of_threads; //no of blocks assuming roundoff that c causes

//for time measurements to avoid getting wrong results due to noise as there are other factors we do the kernel execution 50 times

for(int j=0;j<50;j++){
     vecAdd<<<number_of_blocks,number_of_threads>>>(d_a,d_b,d_c,n);
}
     cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

     //optional verification starts
     bool ok=true;
     for(int j=0;j<n;j++){
      if(h_c[j]-3.0f>1e-5){
        ok=false;
        break;
      }
     }

     printf(ok ? "Verified successfully" : "failed");

     //optional verification ends
     free(h_a);
     free(h_b);
     free(h_c);

     cudaFree(d_a);
     cudaFree(d_b);
     cudaFree(d_c);

     return 0;


}

Overwriting vector_add.cu


In [ ]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector_add

Verified successfully

Vector add kernel with warp divergence

In [ ]:
%%writefile vector_add_divergence.cu
#include <cstdio>
#include <cstdlib>

__global__ void vecAdd(const float *a,const float *b, float *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
  if (i < n && i%2==0) {
    c[i] = a[i] + 2*b[i];
  }
  else if(i<n&& i%2!=0){
    c[i]=a[i]+b[i];
  }
}

int main(){
     int n = 1<<20;
     size_t bytes = n * sizeof(float);
     float *h_a, *h_b, *h_c;
     float *d_a, *d_b, *d_c;

     h_a = (float *)malloc(bytes);
     h_b = (float *)malloc(bytes);
     h_c = (float *)malloc(bytes);


     for(int i=0;i<n;i++){
       h_a[i]=1;
       h_b[i]=2;
     }

     cudaMalloc(&d_a, bytes);
     cudaMalloc(&d_b, bytes);
     cudaMalloc(&d_c, bytes);


     cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
     cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);


     int number_of_threads=1024; //per block threads
     int number_of_blocks=(n+number_of_threads-1)/number_of_threads; //no of blocks assuming roundoff that c causes


//for time measurements to avoid getting wrong results due to noise as there are other factors we do the kernel execution 50 times

for(int j=0;j<50;j++){
     vecAdd<<<number_of_blocks,number_of_threads>>>(d_a,d_b,d_c,n);
}
     cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

     //optional verification starts
     bool ok=true;
     for(int j=0;j<n;j++){
      if(j%2!=0&&h_c[j]-3.0f>1e-5){
        ok=false;
        break;
      }
      else if(j%2==0&&h_c[j]-5.0f>1e-5){
        ok=false;
        break;
      }
     }

     printf(ok ? "Verified successfully" : "failed");

     //optional verification ends
     free(h_a);
     free(h_b);
     free(h_c);

     cudaFree(d_a);
     cudaFree(d_b);
     cudaFree(d_c);

     return 0;


}

Overwriting vector_add_divergence.cu


In [ ]:
!nvcc vector_add_divergence.cu -o vector_add_divergence

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector_add_divergence

Verified successfully

We need to run the programs many times before profiling

In [ ]:
!time ./vector_add

Verified successfully
real	0m2.465s
user	0m0.028s
sys	0m0.370s


In [ ]:
!time ./vector_add_divergence

Verified successfully
real	0m2.599s
user	0m0.042s
sys	0m0.384s


This measurement proves warp divergence case the total time taken from start to end (real) is more similarly more time to execute user space code by CPU(user) and OS code (sys) by CPU but this includes a lot of factors like
program startup
+ CUDA initialization
+ cudaMalloc
+ cudaMemcpy
+ kernel execution
+ synchronization
+ cudaFree
+ other overhead

so we dive bit deeper using NVIDIA's total system profiler nsys

In [ ]:
!sudo apt-get update
!sudo apt-get install -y nsight-systems-2026.1.3

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,909 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127

In [ ]:
!nsys profile --trace=cuda -o normal_profile ./vector_add

Verified successfullyGenerating '/tmp/nsys-report-a8d6.qdstrm'
[1/1] [========================100%] normal_profile.nsys-rep
Generated:
	/content/drive/MyDrive/cuda-project/normal_profile.nsys-rep


In [ ]:
!nsys profile --trace=cuda -o divergence_profile ./vector_add_divergence

Verified successfullyGenerating '/tmp/nsys-report-b54f.qdstrm'
Failed to create '/content/drive/MyDrive/cuda-project/divergence_profile.nsys-rep': File exists.
Use `--force-overwrite true` to overwrite existing files.
[1/1] [========================100%] nsys-report-442e.nsys-rep
Generated:
	/tmp/nsys-report-442e.nsys-rep


In [ ]:
!nsys stats normal_profile.nsys-rep

Generating SQLite file normal_profile.sqlite from normal_profile.nsys-rep
Processing [normal_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/nvtx_sum.py]... 
SKIPPED: normal_profile.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [normal_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/osrt_sum.py]... 
SKIPPED: normal_profile.sqlite does not contain OS Runtime trace data.

Processing [normal_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/cuda_api_sum.py]... 

 ** CUDA API Summary (cuda_api_sum):

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)    Med (ns)   Min (ns)   Max (ns)     StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  ---------  --------  -----------  -------------  ----------------------
     95.4      173,596,232          3  57,865,410.7   74,872.0    59,882  173,461,478  100,109,131.2  cudaMalloc           

In [ ]:
!nsys stats divergence_profile.nsys-rep


NOTICE: Existing SQLite export found: divergence_profile.sqlite
        It is assumed file was previously exported from: divergence_profile.nsys-rep
        Consider using --force-export=true if needed.

Processing [divergence_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/nvtx_sum.py]... 
SKIPPED: divergence_profile.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [divergence_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/osrt_sum.py]... 
SKIPPED: divergence_profile.sqlite does not contain OS Runtime trace data.

Processing [divergence_profile.sqlite] with [/opt/nvidia/nsight-systems/2026.1.3/target-linux-x64/reports/cuda_api_sum.py]... 

 ** CUDA API Summary (cuda_api_sum):

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)     Med (ns)    Min (ns)    Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  ------------  -----------  ---------  -----------  -----

Now analysing the GPU kernel summary only the average time for normal case is 54 micro secs and and for divergent case is 60 micro secs but lets not be that quick in judging this as this includes compute+memory access instructions.

In [ ]:
!rm -f vector_add_ncu.ncu-rep

!ncu --launch-skip 49 --launch-count 1 \
  --metrics "smsp__sass_inst_executed_op_branch,smsp__sass_branch_targets_threads_divergent,smsp__sass_branch_targets_threads_uniform,smsp__sass_average_branch_targets_threads_uniform" \
  -o vector_add_ncu \
  ./vector_add


==PROF== Connected to process 21913 (/content/drive/MyDrive/cuda-project/vector_add)
==PROF== Profiling "vecAdd" - 0 (1/1): 0%....50%....100% - 1 pass
Verified successfully==PROF== Disconnected from process 21913
==PROF== Report: /content/drive/MyDrive/cuda-project/vector_add_ncu.ncu-rep


In [ ]:
!rm -f divergence_ncu.ncu-rep

!ncu --launch-skip 49 --launch-count 1 \
  --metrics "smsp__sass_inst_executed_op_branch,smsp__sass_branch_targets_threads_divergent,smsp__sass_branch_targets_threads_uniform,smsp__sass_average_branch_targets_threads_uniform" \
  -o divergence_ncu \
  ./vector_add_divergence

==PROF== Connected to process 22256 (/content/drive/MyDrive/cuda-project/vector_add_divergence)
==PROF== Profiling "vecAdd" - 0 (1/1): 0%....50%....100% - 1 pass
Verified successfully==PROF== Disconnected from process 22256
==PROF== Report: /content/drive/MyDrive/cuda-project/divergence_ncu.ncu-rep


In [ ]:
!ncu --import vector_add_ncu.ncu-rep

[21913] vector_add@127.0.0.1
  vecAdd(const float *, const float *, float *, int) (1024, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------------------------------------- ----------- ------------
    Metric Name                                                Metric Unit Metric Value
    ---------------------------------------------------------- ----------- ------------
    smsp__sass_average_branch_targets_threads_uniform.max_rate                        1
    smsp__sass_average_branch_targets_threads_uniform.pct                %            0
    smsp__sass_average_branch_targets_threads_uniform.ratio                           0
    smsp__sass_branch_targets_threads_divergent.avg                                   0
    smsp__sass_branch_targets_threads_divergent.max                                   0
    smsp__sass_branch_targets_threads_divergent.min                                   0
    smsp__sass_bra

In [ ]:
!ncu --import divergence_ncu.ncu-rep

[22256] vector_add_divergence@127.0.0.1
  vecAdd(const float *, const float *, float *, int) (1024, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------------------------------------- ----------- ------------
    Metric Name                                                Metric Unit Metric Value
    ---------------------------------------------------------- ----------- ------------
    smsp__sass_average_branch_targets_threads_uniform.max_rate                        1
    smsp__sass_average_branch_targets_threads_uniform.pct                %            0
    smsp__sass_average_branch_targets_threads_uniform.ratio                           0
    smsp__sass_branch_targets_threads_divergent.avg                              204.80
    smsp__sass_branch_targets_threads_divergent.max                                 216
    smsp__sass_branch_targets_threads_divergent.min                                 192
    sms

Cleary shows the absence of branching in normal case whereas presence of branching in warp_divergence case there is clear divergence on all 32,768 warps no two executions have uniform threads, now to prove performance degradation

In [ ]:
!ncu --launch-skip 49 --launch-count 1 \
     --set full \
     -o normal_full \
     ./vector_add

==PROF== Connected to process 29038 (/content/drive/MyDrive/cuda-project/vector_add)
==PROF== Profiling "vecAdd" - 0 (1/1): 0%....50%....100% - 31 passes
Verified successfully==PROF== Disconnected from process 29038
==PROF== Report: /content/drive/MyDrive/cuda-project/normal_full.ncu-rep


In [ ]:
!ncu --launch-skip 49 --launch-count 1 \
     --set full \
     -o divergence_full \
     ./vector_add_divergence

==PROF== Connected to process 29214 (/content/drive/MyDrive/cuda-project/vector_add_divergence)
==PROF== Profiling "vecAdd" - 0 (1/1): 0%....50%....100% - 31 passes
Verified successfully==PROF== Disconnected from process 29214
==PROF== Report: /content/drive/MyDrive/cuda-project/divergence_full.ncu-rep


In [ ]:
!ncu --import normal_full.ncu-rep

[29038] vector_add@127.0.0.1
  vecAdd(const float *, const float *, float *, int) (1024, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.96
    SM Frequency                    Mhz       584.64
    Elapsed Cycles                cycle       30,236
    Memory Throughput                 %        77.73
    DRAM Throughput                   %        77.73
    Duration                         us        51.71
    L1/TEX Cache Throughput           %        27.43
    L2 Cache Throughput               %        27.92
    SM Active Cycles              cycle    24,333.35
    Compute (SM) Throughput           %        21.77
    ----------------------- ----------- ------------

    OPT   Memory is more heavily utilized than Compute: Look at the

In [ ]:
!ncu --import divergence_full.ncu-rep

[29214] vector_add_divergence@127.0.0.1
  vecAdd(const float *, const float *, float *, int) (1024, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.93
    SM Frequency                    Mhz       584.48
    Elapsed Cycles                cycle       38,534
    Memory Throughput                 %        85.03
    DRAM Throughput                   %        85.03
    Duration                         us        65.92
    L1/TEX Cache Throughput           %        47.77
    L2 Cache Throughput               %        29.20
    SM Active Cycles              cycle    32,278.28
    Compute (SM) Throughput           %        30.37
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% 

There is a clear indication of change in average divergent branches from 0 to 204.80, also branch instructions have risen doubled also active threads per warp have fallen from 32 to 23.65 so only 74% of warp efficiency is being used. also the executed instructions have gone up from 491,520 of normal to 753,664 showing more instructions being used (this redundant work is surely contributing to slow down).

There is also an increase of duration from 51.71 micro secs from normal to 65.92 micro secs for divergence and number of elapsed cycle from 30,236 of normal to 38,534 for divergent.

these cycles are only considering the execution of the GPU kernel and not copy between CPU GPU so are good enough to decide on performance.

There was a predication on first normal case due to a simple one condition thing to apply if(i<n) but in second case predication was very expensive so branching was chosen and this significantly affects the performance




In [24]:
!git add vector_add.cu
!git add vector_add_divergence.cu


In [26]:
!git config --global user.email "chiragnanda81@gmail.com"
!git config --global user.name "Chirag Nanda"
!git commit -m "phase 1 work to measure divergence"

[main (root-commit) 959006a] phase 1 work to measure divergence
 2 files changed, 146 insertions(+)
 create mode 100644 vector_add.cu
 create mode 100644 vector_add_divergence.cu


In [36]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git push https://{token}@github.com/darklord-09/my_cuda_project.git

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 1.02 KiB | 209.00 KiB/s, done.
Total 4 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), done.
To https://github.com/darklord-09/my_cuda_project.git
 * [new branch]      main -> main


In [38]:
%%writefile global_latency.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define ARRAY_SIZE (1<<24)

__global__ void global_cycle_counter(unsigned int * data, unsigned int * sink, long long * cycles, long long int iterations){
   unsigned int idx=0;
   long long start=clock64();
   for(int i=0;i<iterations;i++){
    idx=data[idx];
   }
   long long end=clock64();
   *cycles=end-start;
   *sink=idx;
}

int main(){
  unsigned int *h_data = (unsigned int *) malloc(ARRAY_SIZE*sizeof(unsigned int));
  unsigned int *h_sink= (unsigned int *) malloc(sizeof(unsigned int));
  unsigned int *d_data;
  unsigned int *d_sink;
  long long *h_cycles = (long long *) malloc(sizeof(long long));
  long long *d_cycles;

  for(int i=0;i<ARRAY_SIZE;i++){
    h_data[i]=i;
  }

  srand(42);

  for(int i=ARRAY_SIZE-1;i>0;i--){
    int j=rand()%(i+1);
    int temp=h_data[i];
    h_data[i]=h_data[j];
    h_data[j]=temp;
  }

  cudaMalloc(&d_data,ARRAY_SIZE*sizeof(unsigned int));
  cudaMalloc(&d_sink,sizeof(unsigned int));
  cudaMalloc(&d_cycles,sizeof(long long));

  cudaMemcpy(d_data,h_data,ARRAY_SIZE*sizeof(unsigned int),cudaMemcpyHostToDevice);

  global_cycle_counter<<<1,1>>>(d_data,d_sink,d_cycles,ARRAY_SIZE);
  cudaDeviceSynchronize();

  cudaMemcpy(h_cycles,d_cycles,sizeof(long long),cudaMemcpyDeviceToHost);



  printf("Global memory: %.2f cycles/access\n", (double)*h_cycles / ARRAY_SIZE);

  cudaFree(d_data);
  cudaFree(d_sink);
  cudaFree(d_cycles);
  free(h_data);
  free(h_sink);
  free(h_cycles);


return 0;

}

Writing global_latency.cu


In [ ]:
!nvcc global_latency.cu -o global_latency

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
! ./global_latency

Global memory: 490.10 cycles/access


In [39]:
%%writefile shared_latency.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define S_MEM 1024

__global__ void shared_threads_opearations(unsigned int * seed, long long * cycles, int iterations){
   __shared__ unsigned int s_data[S_MEM];
   for(int i=threadIdx.x;i<S_MEM;i+=blockDim.x){
      s_data[i]=seed[i];
   }
   __syncthreads();

   if(threadIdx.x==0){
      unsigned int idx=0;
      long long start=clock64();
      for(int i=0;i<iterations;i++){
         idx=s_data[idx%S_MEM];
      }
      long long end=clock64();
      *cycles=end-start;
      seed[0]=idx;
   }
}

int main(){

    unsigned int *h_seed = (unsigned int *) malloc(S_MEM*sizeof(unsigned int));
  unsigned int *d_seed;
  long long *h_cycles = (long long *) malloc(sizeof(long long));
  long long *d_cycles;

  cudaMalloc(&d_seed,sizeof(unsigned int));
  cudaMalloc(&d_cycles,sizeof(long long));

  srand(7);

  for(int i=S_MEM-1;i>0;i--){
    int j=rand()%(i+1);
    int temp=h_seed[i];
    h_seed[i]=h_seed[j];
    h_seed[j]=temp;}

    cudaMemcpy(d_seed,h_seed,sizeof(unsigned int),cudaMemcpyHostToDevice);

  shared_threads_opearations<<<1,256>>>(d_seed,d_cycles,S_MEM);

  cudaDeviceSynchronize();

  cudaMemcpy(h_cycles,d_cycles,sizeof(long long),cudaMemcpyDeviceToHost);



  printf("Shared memory: %.2f cycles/access\n", (double)*h_cycles / S_MEM);

  cudaFree(d_seed);
  cudaFree(d_cycles);
  free(h_seed);
  free(h_cycles);


return 0;
}

Overwriting shared_latency.cu


In [ ]:
!nvcc shared_latency.cu -o shared_latency

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
! ./shared_latency

Shared memory: 35.19 cycles/access


In [ ]:
%%writefile broadcast_scatter.cu
#include <cstdio>
#include <cuda_runtime.h>

__constant__ float c_arr[256];

__global__ void broadcast_run(float * out,int n, int iterations){
   int i=blockIdx.x*blockDim.x+threadIdx.x;

   if(i<n){
      float num=0.0f;
      #pragma unroll 1
      for(int j=0;j<iterations;j++){
         num+=c_arr[0];
      }
      out[i]=num;
   }
}

__global__ void scatter_run(float * out,int n, int iterations){
   int i=blockIdx.x*blockDim.x+threadIdx.x;

   if(i<n){
      float num=0.0f;
      int lane=threadIdx.x;
      #pragma unroll 1
      for(int j=0;j<iterations;j++){
         num+=c_arr[(32*j+lane)%256];
      }
      out[i]=num;
   }
}

int main(){
   float h_arr[256];
   for(int j=0;j<256;j++){
    h_arr[j]=(float)j+0.001f;
   }
   cudaMemcpyToSymbol(c_arr,h_arr,sizeof(float)*256);

   float * d_out;
   int n=1<<20;
   size_t bytes=n*sizeof(float);

   cudaMalloc(&d_out,bytes);

   int threads=32;
   int blocks=(n+threads-1)/threads;
   int iterations=2000;

//warm up run to avoid noise
   broadcast_run<<<blocks, threads>>>(d_out, n, iterations);
cudaDeviceSynchronize();


   cudaEvent_t start,stop;
   cudaEventCreate(&start);
   cudaEventCreate(&stop);

   cudaEventRecord(start);
   broadcast_run<<<blocks,threads>>>(d_out,n,iterations);
   cudaEventRecord(stop);

   cudaEventSynchronize(stop);
   float broadcast_time;
   cudaEventElapsedTime(&broadcast_time,start,stop);

   cudaEventRecord(start);
   scatter_run<<<blocks,threads>>>(d_out,n,iterations);
   cudaEventRecord(stop);

   cudaEventSynchronize(stop);
   float scatter_time;
   cudaEventElapsedTime(&scatter_time,start,stop);

   printf("Broadcast run time %.4f\n", broadcast_time);
   printf("Scatter run time %.4f\n", scatter_time);
   printf("Slowdown time %.4f", scatter_time/broadcast_time);

   cudaFree(d_out);
   return 0;
}




Overwriting broadcast_scatter.cu


In [ ]:
!nvcc broadcast_scatter.cu -o broadcast_scatter

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
! ./broadcast_scatter

Broadcast run time 3.3474
Scatter run time 130.3491
Slowdown time 38.9409

The broadcast runtime was being shown higher than closer to scatter time due to compiler optimisation due to the regular formula threadIdx%32 as it is predictable access pattern across warps in a block so there might be a predictive optimisation ,
also due to too many warps in a block the GPU scheduler can be overlapping warps to hide execution latency. Now we have increased the const array size and made the access more random and the result was starkly visible. Also 1 KB is such a small cache that once stored it can be used to make the time difference neglible. But let us run some experiments to confirm this.

In [ ]:
%%writefile broadcast_scatter_exp.cu
#include <cstdio>
#include <cuda_runtime.h>

__constant__ float c_arr[256];

__global__ void broadcast_run(float * out,int n, int iterations){
   int i=blockIdx.x*blockDim.x+threadIdx.x;

   if(i<n){
      float num=0.0f;
      #pragma unroll 1
      for(int j=0;j<iterations;j++){
         num+=c_arr[0];
      }
      out[i]=num;
   }
}

__global__ void scatter_run(float * out,int n, int iterations){
   int i=blockIdx.x*blockDim.x+threadIdx.x;

   if(i<n){
      float num=0.0f;
      int lane=threadIdx.x;
      #pragma unroll 1
      for(int j=0;j<iterations;j++){
         num+=c_arr[lane%32];
      }
      out[i]=num;
   }
}

int main(){
   float h_arr[256];
   for(int j=0;j<256;j++){
    h_arr[j]=(float)j+0.001f;
   }
   cudaMemcpyToSymbol(c_arr,h_arr,sizeof(float)*256);

   float * d_out;
   int n=1<<20;
   size_t bytes=n*sizeof(float);

   cudaMalloc(&d_out,bytes);

   int threads=32;
   int blocks=(n+threads-1)/threads;
   int iterations=2000;

//warm up run to avoid noise
   broadcast_run<<<blocks, threads>>>(d_out, n, iterations);

cudaDeviceSynchronize();


   cudaEvent_t start,stop;
   cudaEventCreate(&start);
   cudaEventCreate(&stop);

   cudaEventRecord(start);
   broadcast_run<<<blocks,threads>>>(d_out,n,iterations);
   cudaEventRecord(stop);

   cudaEventSynchronize(stop);
   float broadcast_time;
   cudaEventElapsedTime(&broadcast_time,start,stop);

   cudaEventRecord(start);
   scatter_run<<<blocks,threads>>>(d_out,n,iterations);
   cudaEventRecord(stop);

   cudaEventSynchronize(stop);
   float scatter_time;
   cudaEventElapsedTime(&scatter_time,start,stop);

   printf("Broadcast run time %.4f\n", broadcast_time);
   printf("Scatter run time %.4f\n", scatter_time);
   printf("Slowdown time %.4f", scatter_time/broadcast_time);

   cudaFree(d_out);
   return 0;
}




Overwriting broadcast_scatter_exp.cu


In [ ]:
! nvcc broadcast_scatter_exp.cu -o broadcast_scatter_exp

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
! ./broadcast_scatter_exp

Broadcast run time 3.6466
Scatter run time 3.0247
Slowdown time 0.8295

Notes : cache size does not matter with 256 elements and the similar acces pattern same order of change

number of threads showed no correlation with the new access pattern

The only issue was using lane id for access across warps in a block.



In [ ]:
!ncu --set full ./broadcast_scatter_exp

==PROF== Connected to process 5409 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 31 passes
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 31 passes
Broadcast run time 7276.3828
Scatter run time 7358.7847
Slowdown time 1.0113==PROF== Disconnected from process 5409
[5409] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle    3,004,124
    Memory Throughput                 %         0.24
    DRAM T

In [ ]:
!ncu --set full ./broadcast_scatter

==PROF== Connected to process 5731 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 31 passes
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 31 passes
Broadcast run time 7297.6763
Scatter run time 13884.2285
Slowdown time 1.9026==PROF== Disconnected from process 5731
[5731] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       584.99
    Elapsed Cycles                cycle    3,001,920
    Memory Throughput                 %         0.24
    DRAM Throughp

Comparable in instruction executions as expected as the differnece will be in memory access IDC

In [ ]:
!ncu --query-metrics | grep -i -E "idc|constant|ldc"

idc__cycles_active                                                          Counter         cycle           # of cycles where IDC was active                                      
idc__cycles_elapsed                                                         Counter         cycle           # of cycles elapsed on IDC                                            
idc__cycles_in_frame                                                        Counter         cycle           # of cycles in user-defined frame                                     
idc__cycles_in_region                                                       Counter         cycle           # of cycles in user-defined region                                    
idc__request_cycles_active                                                  Counter         cycle           # of cycles where IDC processed requests from SM                      
idc__request_hit_rate                                                       Ratio                        

In [ ]:
!ncu --metrics idc__requests ./broadcast_scatter

==PROF== Connected to process 6267 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 35.8784
Scatter run time 241.8237
Slowdown time 6.7401==PROF== Disconnected from process 6267
[6267] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    idc__requests.avg                        0
    idc__requests.max                        0
    idc__requests.min                        0
    idc__requests.sum                        0
    ----------------- ----------- ------------

  broadcast_run(float *,

In [ ]:
!ncu --metrics idc__requests_lookup_hit ./broadcast_scatter

==PROF== Connected to process 6847 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 33.7196
Scatter run time 242.8098
Slowdown time 7.2009==PROF== Disconnected from process 6847
[6847] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    idc__requests_lookup_hit.avg                        0
    idc__requests_lookup_hit.max                        0
    idc__requests_lookup_hit.min                        0
    idc__requests_lookup_hit.sum                     

In [ ]:
!ncu --metrics idc__requests_lookup_miss ./broadcast_scatter

==PROF== Connected to process 6927 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 32.7783
Scatter run time 245.4280
Slowdown time 7.4875==PROF== Disconnected from process 6927
[6927] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------------------- ----------- ------------
    Metric Name                   Metric Unit Metric Value
    ----------------------------- ----------- ------------
    idc__requests_lookup_miss.avg                        0
    idc__requests_lookup_miss.max                        0
    idc__requests_lookup_miss.min                        0
    idc__requests_lookup_miss.sum              

In [ ]:
!ncu --metrics idc__request_hit_rate ./broadcast_scatter

==PROF== Connected to process 6988 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 32.3353
Scatter run time 248.0010
Slowdown time 7.6697==PROF== Disconnected from process 6988
[6988] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ------------------------------ ----------- ------------
    Metric Name                    Metric Unit Metric Value
    ------------------------------ ----------- ------------
    idc__request_hit_rate.max_rate                        1
    idc__request_hit_rate.pct                %            0
    idc__request_hit_rate.ratio                           0
    ------------------------------ ------

In [ ]:
!ncu --metrics idc__requests ./broadcast_scatter_exp

==PROF== Connected to process 7106 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 31.8986
Scatter run time 31.6305
Slowdown time 0.9916==PROF== Disconnected from process 7106
[7106] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------- ----------- ------------
    Metric Name       Metric Unit Metric Value
    ----------------- ----------- ------------
    idc__requests.avg                        0
    idc__requests.max                        0
    idc__requests.min                        0
    idc__requests.sum                        0
    ----------------- ----------- ------------

  broadcast_run(f

In [ ]:
!ncu --metrics idc__requests_lookup_hit ./broadcast_scatter_exp

==PROF== Connected to process 10975 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 36.5995
Scatter run time 35.1660
Slowdown time 0.9608==PROF== Disconnected from process 10975
[10975] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    idc__requests_lookup_hit.avg                        0
    idc__requests_lookup_hit.max                        0
    idc__requests_lookup_hit.min                        0
    idc__requests_lookup_hit.sum           

In [ ]:
!ncu --metrics idc__requests_lookup_miss ./broadcast_scatter_exp

==PROF== Connected to process 7166 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 31.8844
Scatter run time 32.7647
Slowdown time 1.0276==PROF== Disconnected from process 7166
[7166] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------------------- ----------- ------------
    Metric Name                   Metric Unit Metric Value
    ----------------------------- ----------- ------------
    idc__requests_lookup_miss.avg                        0
    idc__requests_lookup_miss.max                        0
    idc__requests_lookup_miss.min                        0
    idc__requests_lookup_miss.sum       

Cache miss is not the slowness issue almost similar cache hits

In [ ]:
!ncu --metrics idc__request_hit_rate ./broadcast_scatter_exp

==PROF== Connected to process 7229 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 32.3951
Scatter run time 32.7305
Slowdown time 1.0104==PROF== Disconnected from process 7229
[7229] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ------------------------------ ----------- ------------
    Metric Name                    Metric Unit Metric Value
    ------------------------------ ----------- ------------
    idc__request_hit_rate.max_rate                        1
    idc__request_hit_rate.pct                %            0
    idc__request_hit_rate.ratio                           0
    ------------------------------

In [ ]:
!ncu --metrics sm__idc_divergent_instruction_replays ./broadcast_scatter

==PROF== Connected to process 9537 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 263.0031
Scatter run time 471.4263
Slowdown time 1.7925==PROF== Disconnected from process 9537
[9537] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------------------------------- ----------- ------------
    Metric Name                               Metric Unit Metric Value
    ----------------------------------------- ----------- ------------
    sm__idc_divergent_instruction_replays.avg        inst            0
    sm__idc_divergent_instruction_replays.max        inst            0
    sm__idc_divergent_instruction_replays.min

In [ ]:
!ncu --metrics sm__idc_divergent_instruction_replays ./broadcast_scatter_exp

==PROF== Connected to process 9652 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 267.6278
Scatter run time 277.4543
Slowdown time 1.0367==PROF== Disconnected from process 9652
[9652] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ----------------------------------------- ----------- ------------
    Metric Name                               Metric Unit Metric Value
    ----------------------------------------- ----------- ------------
    sm__idc_divergent_instruction_replays.avg        inst            0
    sm__idc_divergent_instruction_replays.max        inst            0
    sm__idc_divergent_instruction_rep

In [ ]:
!ncu --metrics sm__idc_divergent_instructions ./broadcast_scatter

==PROF== Connected to process 5155 (/content/drive/MyDrive/cuda-project/broadcast_scatter)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 260.7034
Scatter run time 469.2354
Slowdown time 1.7999==PROF== Disconnected from process 5155
[5155] broadcast_scatter@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------------- ----------- ------------
    Metric Name                        Metric Unit Metric Value
    ---------------------------------- ----------- ------------
    sm__idc_divergent_instructions.avg        inst            0
    sm__idc_divergent_instructions.max        inst            0
    sm__idc_divergent_instructions.min        inst            0
    sm__idc_dive

In [ ]:
!ncu --metrics sm__idc_divergent_instructions ./broadcast_scatter_exp



==PROF== Connected to process 6802 (/content/drive/MyDrive/cuda-project/broadcast_scatter_exp)
==PROF== Profiling "broadcast_run" - 0: 0%....50%....100% - 1 pass
==PROF== Profiling "broadcast_run" - 1: 0%....50%....100% - 1 pass
==PROF== Profiling "scatter_run(float *, int, int)" - 2: 0%....50%....100% - 1 pass
Broadcast run time 263.2906
Scatter run time 261.4411
Slowdown time 0.9930==PROF== Disconnected from process 6802
[6802] broadcast_scatter_exp@127.0.0.1
  broadcast_run(float *, int, int) (32768, 1, 1)x(32, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: Command line profiler metrics
    ---------------------------------- ----------- ------------
    Metric Name                        Metric Unit Metric Value
    ---------------------------------- ----------- ------------
    sm__idc_divergent_instructions.avg        inst            0
    sm__idc_divergent_instructions.max        inst            0
    sm__idc_divergent_instructions.min        inst            0
    sm__

The divergent IDC instructions (memory access instructions) in the broadcast_scatter_exp case is 2000 times lower this shows that the in the lane%32 as the address accesses for threads were loop invariant the compiler took it out of loop as seen in SASS so less IDC accesses but in broadcast_scatter case the access is loop dependent depends on j so everytime there was a different access. This idc_divergence is the issue that occurs in scattered access and broadcast case there is 0 idc_divergent access.

In [ ]:
!cuobjdump --dump-sass ./broadcast_scatter_exp


Fatbin elf code:
arch = sm_52
code version = [1,7]
host = linux
compile_size = 64bit

	code for sm_52

Fatbin elf code:
arch = sm_52
code version = [1,7]
host = linux
compile_size = 64bit

	code for sm_52
		Function : _Z11scatter_runPfii
	.headerflags	@"EF_CUDA_TEXMODE_UNIFIED EF_CUDA_64BIT_ADDRESS EF_CUDA_SM52 EF_CUDA_VIRTUAL_SM(EF_CUDA_SM52)"
                                                                                 /* 0x001cfc00e22007f6 */
        /*0008*/                   MOV R1, c[0x0][0x20] ;                        /* 0x4c98078000870001 */
        /*0010*/                   S2R R2, SR_CTAID.X ;                          /* 0xf0c8000002570002 */
        /*0018*/                   S2R R5, SR_TID.X ;                            /* 0xf0c8000002170005 */
                                                                                 /* 0x001fd842fec20ff1 */
        /*0028*/                   XMAD.MRG R3, R2.reuse, c[0x0] [0x8].H1, RZ ;  /* 0x4f107f8000270203 */
        /*0030*/

still the question why is scatter faster than broadcast in the exp run. Claude suggests improper warmup and suggested some changes

In [ ]:
%%writefile broadcast_scatter_rigorous.cu
#include <cstdio>
#include <cmath>
#include <cuda_runtime.h>
__constant__ float c_arr[256];

__global__ void broadcast_run(float* out, int n, int iterations) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float num = 0.0f;
        #pragma unroll 1
        for (int j = 0; j < iterations; j++) num += c_arr[0];
        out[i] = num;
    }
}
__global__ void scatter_run(float* out, int n, int iterations) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        float num = 0.0f;
        int lane = threadIdx.x;
        #pragma unroll 1
        for (int j = 0; j < iterations; j++) num += c_arr[lane % 32];
        out[i] = num;
    }
}

int main() {
    float h_arr[256];
    for (int j = 0; j < 256; j++) h_arr[j] = (float)j + 0.001f;
    cudaMemcpyToSymbol(c_arr, h_arr, sizeof(float) * 256);

    float* d_out;
    int n = 1 << 20, threads = 32, iterations = 2000;
    int blocks = (n + threads - 1) / threads;
    cudaMalloc(&d_out, n * sizeof(float));

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    const int WARMUP = 5, RUNS = 20;

    // Multiple warm-up launches, BOTH kernels, before any timing starts
    for (int w = 0; w < WARMUP; w++) {
        broadcast_run<<<blocks, threads>>>(d_out, n, iterations);
        scatter_run<<<blocks, threads>>>(d_out, n, iterations);
    }
    cudaDeviceSynchronize();

    // Interleaved: B, S, B, S, ... so any residual drift hits both evenly
    float tB[RUNS], tS[RUNS];
    for (int i = 0; i < RUNS; i++) {
        cudaEventRecord(start);
        broadcast_run<<<blocks, threads>>>(d_out, n, iterations);
        cudaEventRecord(stop); cudaEventSynchronize(stop);
        cudaEventElapsedTime(&tB[i], start, stop);

        cudaEventRecord(start);
        scatter_run<<<blocks, threads>>>(d_out, n, iterations);
        cudaEventRecord(stop); cudaEventSynchronize(stop);
        cudaEventElapsedTime(&tS[i], start, stop);
    }

    auto stats = [&](float* a, float& mean, float& sd) {
        mean = 0; for (int i = 0; i < RUNS; i++) mean += a[i]; mean /= RUNS;
        sd = 0; for (int i = 0; i < RUNS; i++) sd += (a[i]-mean)*(a[i]-mean);
        sd = sqrtf(sd / RUNS);
    };
    float mB, sB, mS, sS;
    stats(tB, mB, sB);
    stats(tS, mS, sS);

    printf("Broadcast: mean=%.4f ms  stddev=%.4f ms\n", mB, sB);
    printf("Scatter:   mean=%.4f ms  stddev=%.4f ms\n", mS, sS);
    printf("Scatter/Broadcast ratio: %.4f\n", mS / mB);

    cudaFree(d_out);
    return 0;
}

Overwriting broadcast_scatter_rigorous.cu


In [ ]:
!nvcc broadcast_scatter_rigorous.cu -o broadcast_scatter_rigorous

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./broadcast_scatter_rigorous

Broadcast: mean=3.5182 ms  stddev=0.0036 ms
Scatter:   mean=3.2767 ms  stddev=0.0021 ms
Scatter/Broadcast ratio: 0.9314


With warmup also same issue so this was the hypothesis drawn from SASS file
-Scatter: after the compiler hoists the loop-invariant lane % 32 address computation and the load itself: one LDC up front, then 2000 iterations of pure register-to-register FADD — about as cheap as a loop body can get.
-Broadcast: because c_arr[0] is a compile-time-constant address, the compiler folds it directly into FADD R0, R0, c[0x3][0x0] as an operand — but that means every single iteration re-reads the constant bank as part of the instruction's operand fetch, rather than paying that cost once.

The core reason: a cache access, even a fast one that hits, always carries the cost of checking "is this data actually here?" — a register read doesn't have that question at all.
Registers are direct-indexed, not looked up. A register number in an instruction (R0, R5) is just a small index into a fixed, flat array of physical storage slots. The hardware decodes that index and selects the corresponding storage location directly — there's no possibility of a "miss," so there's no tag-comparison circuitry involved at all. It's structurally closer to addressing a specific pigeonhole by number than to "looking something up."
A cache — even the constant cache, even when it hits every single time — is associative, not direct-indexed. Every access has to compute which cache line the address maps to, compare a stored tag against the requested address to confirm it's actually the right data (not just a line, but confirmably this line), and only then deliver the value. That tag-check-and-confirm step happens on every access, hit or miss — a cache hit is faster than a miss, but it's never as fast as skipping the lookup machinery entirely, which is what a register read does.

We go for the numerical proof below


In [ ]:
!nvcc -arch=sm_75 -cubin broadcast_scatter_rigorous.cu -o real.cubin


In [ ]:
!cuobjdump --dump-sass real.cubin | grep -B2 -A2 "FADD"


        /*0130*/                   IADD3 R4, R4, 0x1, RZ ;                       /* 0x0000000104047810 */
                                                                                 /* 0x000fe20007ffe0ff */
        /*0140*/                   FADD R5, R0, R5 ;                             /* 0x0000000500057221 */
                                                                                 /* 0x001fc60000000000 */
        /*0150*/                   ISETP.GE.AND P0, PT, R4, c[0x0][0x16c], PT ;  /* 0x00005b0004007a0c */
--
        /*00e0*/                   IADD3 R0, R0, 0x1, RZ ;                       /* 0x0000000100007810 */
                                                                                 /* 0x000fe20007ffe0ff */
        /*00f0*/                   FADD R5, R5, c[0x3][0x0] ;                    /* 0x00c0000005057621 */
                                                                                 /* 0x000fc60000000000 */
        /*0100*/                   ISETP.GE

In [ ]:
!ncu --import /content/drive/MyDrive/cuda-project/b_prof.ncu-rep --page raw --csv > b_raw_metrics.csv
!ncu --import /content/drive/MyDrive/cuda-project/s_prof.ncu-rep  --page raw --csv > s_raw_metrics.csv

In [ ]:
!grep -i "stall" b_raw_metrics.csv | head -n 20

"ID","Process ID","Process Name","Host Name","Kernel Name","Context","Stream","Block Size","Grid Size","Device","CC","FBPA.TriageA.dramc__read_throughput.avg.pct_of_peak_sustained_elapsed","FBPA.TriageA.dramc__throughput.avg.pct_of_peak_sustained_elapsed","FBPA.TriageA.dramc__write_throughput.avg.pct_of_peak_sustained_elapsed","SM.TriageA.l1tex__data_pipe_lsu_wavefronts.avg","SM.TriageA.l1tex__lsu_writeback_active.avg","SM.TriageA.sm__cycles_active.avg","SM.TriageA.sm__inst_executed_realtime.avg.per_cycle_active","TPC.TriageA.tpc__warps_active_realtime.avg.per_cycle_active","TPC.TriageA.tpc__warps_active_realtime.sum.per_cycle_active","TriageA.l1tex__t_sector_hit_rate_realtime.pct","c2clink__enabled_mask","c2clink__present","derived__avg_thread_executed","derived__avg_thread_executed_true","derived__l1tex__lsu_writeback_bytes_mem_lg.sum.peak_sustained","derived__l1tex__lsu_writeback_bytes_mem_lg.sum.per_second","derived__lts__lts2xbar_bytes.sum.peak_sustained","derived__lts__lts2xbar_b

In [ ]:
import pandas as pd

# Load the raw metrics CSV files
df_b = pd.read_csv('b_raw_metrics.csv')
df_s = pd.read_csv('s_raw_metrics.csv')

# The actual data is in the last row of the Nsight Compute CSV output
row_b = df_b.iloc[-1]
row_s = df_s.iloc[-1]

# Filter for the exact stall metrics we found in the header
stall_cols = [c for c in df_b.columns if 'issue_stalled' in c and 'ratio' in c]

print(f"{'Stall Type':<25} | {'Broadcast':<10} | {'Scatter':<10} | {'Difference'}")
print("-" * 65)

for c in stall_cols:
    try:
        vb = float(row_b[c])
        vs = float(row_s[c])
        # Clean up the ugly metric name for readability
        name = c.split('_stalled_')[1].replace('_per_issue_active.ratio', '')
        diff = vb - vs

        # Only print if the stall actually consumed cycles
        if vb > 0 or vs > 0:
            print(f"{name:<25} | {vb:<10.3f} | {vs:<10.3f} | {diff:+.3f}")
    except:
        pass

Stall Type                | Broadcast  | Scatter    | Difference
-----------------------------------------------------------------
branch_resolving          | 0.749      | 0.749      | +0.000
dispatch_stall            | 0.000      | 0.000      | -0.000
drain                     | 0.003      | 0.003      | +0.000
imc_miss                  | 0.001      | 0.000      | +0.001
math_pipe_throttle        | 0.160      | 0.111      | +0.048
no_instruction            | 1.069      | 0.639      | +0.430
not_selected              | 0.220      | 0.142      | +0.078
selected                  | 1.000      | 1.000      | +0.000
short_scoreboard          | 0.002      | 0.050      | -0.047
wait                      | 3.999      | 3.998      | +0.000


**CHATGPT GENERATED CONCLUSION (verified)**
The data completely solves the mystery, and the answer is hiding in the no_instruction metric (+0.430 cycles).
While both kernels suffer the exact same 4.0-cycle wait for the math to finish, the Broadcast kernel loses nearly half a cycle per instruction because the warp scheduler is physically missing the next instruction from instruction buffer

1. The Official Definition of the Stall
According to the Nsight Compute Metrics Reference, smsp__average_warps_issue_stalled_no_instruction_per_issue_active is defined strictly as:
"Stall No Instruction: Warp is stalled waiting for one or more instructions to be fetched from the instruction cache."

2. Why Constant Memory Triggers It (The Pipeline Backup)
The Turing architecture features a dedicated Uniform Datapath for handling constant memory (c[bank][offset]). Here is how the official documentation explains instruction flow, which explains your data:
When a warp scheduler attempts to issue your Scatter instruction (FADD R5, R0, R5), all operands are already in registers. The instruction decodes instantly and moves to the execution pipeline.
When the scheduler attempts to issue your Broadcast instruction (FADD R5, R5, c[0x3][0x0]), the decoder must pause to signal the Operand Collector to fetch the value from the Uniform Cache.
Because your loop is incredibly tight and executing millions of times in a row, the Uniform Datapath queue saturates. When the decode/operand-collection stage gets blocked waiting for the cache hardware, the pipeline backs up. The Instruction Fetch unit is forced to halt because there is nowhere to put the next instruction, which the Nsight Compute profiler records as a no_instruction stall for that warp.

In [40]:
%%writefile experimental_bandwith.cu
#include <cstdio>
#include <cuda_runtime.h>

__global__ void simple_copy(const float * in, float * out, int n){
   int i=blockIdx.x*blockDim.x+threadIdx.x;
   if(i<n){
      out[i]=in[i];
   }
}

int main(){
  int n=1<<26;
  size_t bytes=n*sizeof(float);
  float * d_in;
  float * d_out;
  float * h_in = (float*)malloc(bytes);

  for(int i=0;i<n;i++){
    h_in[i]=1.0f;
  }


  cudaMalloc(&d_in,bytes);
  cudaMalloc(&d_out,bytes);

  cudaMemcpy(d_in,h_in,bytes,cudaMemcpyHostToDevice);

  int threads=1024;
  int blocks=(n+threads-1)/threads;

  //warm-up run
  simple_copy<<<blocks,threads>>>(d_in,d_out,n);
  cudaDeviceSynchronize();

  cudaEvent_t start,stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  cudaEventRecord(start);
  simple_copy<<<blocks,threads>>>(d_in,d_out,n);
  cudaEventRecord(stop);

  cudaEventSynchronize(stop);
  float time;
  cudaEventElapsedTime(&time,start,stop);

  double GB_processed = (2 * n)/(1e9); //2 as one for read and one for write
  printf("measured bandwidth in GB/s %.4f",GB_processed/(time/1000));

  return 0;
}


Writing experimental_bandwith.cu


In [ ]:
! nvcc experimental_bandwith.cu -o experimental_bandwith

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
! ./experimental_bandwith

measured bandwidth in GB/s 62.5931

In [20]:
%%writefile transponse_matrix_minimal.cu
#include<cstdio>
#include<cmath>
#include<cuda_runtime.h>

__global__ void transposer(float *A,float *B,int N){
  int col=blockIdx.x*blockDim.x+threadIdx.x;
  int row=blockIdx.y*blockDim.y+threadIdx.y;

  if(row<N&&col<N){
    B[col*N+row]=A[row*N+col];
  }
}

int main(){
    float * h_A;
    float * h_B;
    float * d_A;
    float * d_B;
    int N=4096;
    h_A=(float*)malloc(N*N*sizeof(float));
    h_B=(float*)malloc(N*N*sizeof(float));
    cudaMalloc(&d_A,N*N*sizeof(float));
    cudaMalloc(&d_B,N*N*sizeof(float));
    for(int i=0;i<N*N;i++){
        h_A[i]=(float)i;
    }
    cudaMemcpy(d_A,h_A,N*N*sizeof(float),cudaMemcpyHostToDevice);

    dim3 block(32,32);
    dim3 grid((N+block.x-1)/block.x,(N+block.y-1)/block.y);

    //warm up run
    transposer<<<grid,block>>>(d_A,d_B,N);
    cudaDeviceSynchronize();

    cudaEvent_t start,stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    transposer<<<grid,block>>>(d_A,d_B,N);
    cudaEventRecord(stop);

    cudaEventSynchronize(stop);
    cudaMemcpy(h_B,d_B,N*N*sizeof(float),cudaMemcpyDeviceToHost);
    float elapsed_time;
    cudaEventElapsedTime(&elapsed_time,start,stop);
    cudaDeviceSynchronize();


    bool correct=true;
    for(int i=0;i<N;i++){
        for(int j=0;j<N;j++){
            int A_val=h_A[i*N+j];
            int expected_B=h_B[j*N+i];

            if(A_val!=expected_B){
              printf("mistake in row = %d col = %d",i,j);
              correct=false;
              break;
            }
        }
    }

    if(correct){
      printf("the transponse worked correctly with time taken = %f cycles",elapsed_time);
    }

    cudaFree(d_A);
    cudaFree(d_B);

    free(h_A);
    free(h_B);

    return 0;

}

Overwriting transponse_matrix_minimal.cu


In [21]:
!nvcc transponse_matrix_minimal.cu -o transponse_matrix_minimal

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [22]:
! ./transponse_matrix_minimal

the transponse worked correctly with time taken = 1.957440 cycles

In [42]:
! git add global_latency.cu
! git add shared_latency.cu
! git add broadcast_scatter.cu
! git add broadcast_scatter_exp.cu
! git add experimental_bandwith.cu
! git add transponse_matrix_minimal.cu

In [43]:
! git status

On branch main
Your branch is based on 'origin/main', but the upstream is gone.
  (use "git branch --unset-upstream" to fixup)

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   broadcast_scatter.cu
	new file:   broadcast_scatter_exp.cu
	new file:   experimental_bandwith.cu
	new file:   global_latency.cu
	new file:   shared_latency.cu
	new file:   transponse_matrix_minimal.cu

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	b_prof.ncu-rep
	b_raw_metrics.csv
	broadcast_scatter
	broadcast_scatter_exp
	broadcast_scatter_rigorous
	broadcast_scatter_rigorous.cu
	divergence_full.ncu-rep
	divergence_ncu.ncu-rep
	divergence_profile.nsys-rep
	divergence_profile.sqlite
	normal_full.ncu-rep
	normal_profile.nsys-rep
	normal_profile.sqlite
	real.cubin
	s_prof.ncu-rep
	s_raw_metrics.csv
	shared_latency
	transponse_matrix_minimal
	vector_add
	vector_add_divergence
	vector_add_divergence_profile.nsys-rep
	vector_add_ncu.ncu-rep

In [44]:
! git commit -m "phase 2 memory model deep dive"

[main f51ca9c] phase 2 memory model deep dive
 6 files changed, 422 insertions(+)
 create mode 100644 broadcast_scatter.cu
 create mode 100644 broadcast_scatter_exp.cu
 create mode 100644 experimental_bandwith.cu
 create mode 100644 global_latency.cu
 create mode 100644 shared_latency.cu
 create mode 100644 transponse_matrix_minimal.cu


In [46]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git push https://{token}@github.com/darklord-09/my_cuda_project.git

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 3.52 KiB | 451.00 KiB/s, done.
Total 8 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), done.
To https://github.com/darklord-09/my_cuda_project.git
   959006a..f51ca9c  main -> main
